# Estructuras de Datos y Algoritmos para Data Science

**Por que importa:** No vas a implementar un arbol binario en tu trabajo diario. Pero entender COMO funcionan las estructuras de datos te permite:

- Saber por que tu codigo tarda 5 minutos vs 0.5 segundos
- Entender que hace sklearn/pandas por dentro
- Elegir la estructura correcta para cada problema
- Pasar entrevistas tecnicas

---

## 1. Complejidad Computacional: O(n)

Antes de hablar de estructuras, necesitas saber MEDIR la velocidad de un algoritmo.

**Big O** describe como crece el tiempo de ejecucion cuando crece el tamaño de los datos:

| Complejidad | Nombre | Ejemplo | 1,000 datos | 1,000,000 datos |
|---|---|---|---|---|
| O(1) | Constante | Acceder a un elemento por indice | Instantaneo | Instantaneo |
| O(log n) | Logaritmico | Busqueda binaria | 10 ops | 20 ops |
| O(n) | Lineal | Buscar en una lista | 1,000 ops | 1,000,000 ops |
| O(n log n) | Linearitmico | Sorting (merge sort) | 10,000 ops | 20,000,000 ops |
| O(n²) | Cuadratico | Loop dentro de loop | 1,000,000 ops | 10¹² ops (imposible) |
| O(2ⁿ) | Exponencial | Fuerza bruta combinatoria | Imposible | Imposible |

**Regla practica:** Si tu dataset tiene 1 millon de filas, O(n²) tarda horas. O(n log n) tarda segundos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
C_PRIMARY = '#3498db'
C_DANGER = '#e74c3c'
C_SUCCESS = '#2ecc71'
C_DARK = '#2c3e50'
C_ORANGE = '#f39c12'
C_PURPLE = '#9b59b6'

# === Visualizar complejidades ===
n = np.arange(1, 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(n, np.ones_like(n), linewidth=2, label='O(1)', color=C_SUCCESS)
ax.plot(n, np.log2(n), linewidth=2, label='O(log n)', color=C_PRIMARY)
ax.plot(n, n, linewidth=2, label='O(n)', color=C_ORANGE)
ax.plot(n, n * np.log2(n), linewidth=2, label='O(n log n)', color=C_PURPLE)
ax.plot(n, n**2, linewidth=2, label=r'O(n²)', color=C_DANGER)
ax.set_title('Crecimiento de complejidades', fontsize=13, fontweight='bold', color=C_DARK)
ax.set_xlabel('n (datos)')
ax.set_ylabel('Operaciones')
ax.legend()
ax.set_ylim(0, 500)

# Tiempo real: loop vs vectorizacion
sizes = [100, 1000, 5000, 10000, 50000, 100000]
t_loop = []
t_vector = []

for s in sizes:
    arr = np.random.randn(s)
    
    start = time.perf_counter()
    total = 0
    for x in arr:
        total += x**2
    t_loop.append(time.perf_counter() - start)
    
    start = time.perf_counter()
    total = np.sum(arr**2)
    t_vector.append(time.perf_counter() - start)

ax = axes[1]
ax.plot(sizes, t_loop, 'o-', color=C_DANGER, linewidth=2, label='Loop Python')
ax.plot(sizes, t_vector, 'o-', color=C_SUCCESS, linewidth=2, label='NumPy vectorizado')
ax.set_title('Tiempo real: Loop vs Vectorizacion', fontsize=13, fontweight='bold', color=C_DARK)
ax.set_xlabel('n (datos)')
ax.set_ylabel('Tiempo (segundos)')
ax.legend()

plt.tight_layout()
plt.show()

speedup = t_loop[-1] / t_vector[-1]
print(f"Con {sizes[-1]:,} datos: Loop = {t_loop[-1]*1000:.1f}ms, NumPy = {t_vector[-1]*1000:.3f}ms")
print(f"NumPy es {speedup:.0f}x mas rapido. Por eso NUNCA uses loops en datos grandes.")

## 2. Arrays (Listas y NumPy Arrays)

La estructura mas basica y la que mas usas en data science.

| Operacion | Python list | NumPy array | Nota |
|---|---|---|---|
| Acceso por indice | O(1) | O(1) | `arr[5]` |
| Buscar un valor | O(n) | O(n) | Recorre todo |
| Agregar al final | O(1)* | No soporta | `append()` |
| Insertar en medio | O(n) | No soporta | Debe mover elementos |
| Slice | O(k) | O(1)** | NumPy no copia, crea vista |
| Operacion elemento a elemento | O(n) con loop | O(n) vectorizado | NumPy usa C internamente |

**NumPy es rapido porque:**
- Los datos son contiguos en memoria (no punteros dispersos como Python list)
- Las operaciones se ejecutan en C/Fortran compilado, no en Python interpretado
- SIMD: procesa multiples datos en una instruccion del procesador

In [ ]:
# === Arrays: list vs numpy ===
import sys

# Memoria
python_list = list(range(10000))
numpy_array = np.arange(10000)

print("MEMORIA:")
print(f"  Python list (10,000 int): {sys.getsizeof(python_list) + sum(sys.getsizeof(i) for i in python_list[:100])*100:,} bytes aprox")
print(f"  NumPy array (10,000 int): {numpy_array.nbytes:,} bytes")
print(f"  NumPy usa ~{numpy_array.nbytes / (sys.getsizeof(python_list)):,.1f}x menos memoria")

# Operaciones comunes en DS
print("\nOPERACIONES COMUNES:")
arr = np.random.randn(1_000_000)

# Busqueda
start = time.perf_counter()
idx = np.where(arr > 2.0)[0]
t1 = time.perf_counter() - start
print(f"  np.where(arr > 2.0): {t1*1000:.2f}ms — encontro {len(idx):,} elementos")

# Sorting
start = time.perf_counter()
sorted_arr = np.sort(arr)
t2 = time.perf_counter() - start
print(f"  np.sort (1M elementos): {t2*1000:.1f}ms — O(n log n)")

# Slicing (no copia!)
start = time.perf_counter()
view = arr[100:200]  # Esto NO copia datos, es una vista
t3 = time.perf_counter() - start
print(f"  Slice arr[100:200]: {t3*1e6:.2f}us — O(1), es una VISTA, no copia")

## 3. Diccionarios (Hash Maps)

La estructura mas subestimada. Busqueda en **O(1)** — constante, sin importar el tamaño.

| Operacion | List | Dict | Diferencia |
|---|---|---|---|
| Buscar si existe | O(n) | **O(1)** | Dict es n veces mas rapido |
| Acceso por clave | N/A | **O(1)** | |
| Insertar | O(1) append | **O(1)** | |

### Como funciona un hash map?

1. Tomas la clave (ej: "sensor_7")
2. Aplicas una funcion hash: `hash("sensor_7")` → numero grande
3. Ese numero te da la posicion directa en un array interno
4. Acceso directo, sin buscar

**Donde aparece en ML:**
- `pd.DataFrame` usa hash maps internamente para columnas
- `LabelEncoder` / `OrdinalEncoder`: mapeo categoricas → numeros
- Feature hashing (hashing trick) en NLP
- Conteo de frecuencias (Counter)

In [ ]:
# === Dict vs List: busqueda ===
n_elements = [100, 1000, 10000, 100000]
t_list_search = []
t_dict_search = []

for n in n_elements:
    data_list = list(range(n))
    data_dict = {i: True for i in range(n)}
    target = n - 1  # Peor caso: ultimo elemento
    
    start = time.perf_counter()
    for _ in range(1000):
        _ = target in data_list
    t_list_search.append((time.perf_counter() - start) / 1000)
    
    start = time.perf_counter()
    for _ in range(1000):
        _ = target in data_dict
    t_dict_search.append((time.perf_counter() - start) / 1000)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(n_elements, [t*1e6 for t in t_list_search], 'o-', color=C_DANGER, linewidth=2, label='List: O(n)')
ax.plot(n_elements, [t*1e6 for t in t_dict_search], 'o-', color=C_SUCCESS, linewidth=2, label='Dict: O(1)')
ax.set_title('Busqueda: List vs Dict', fontsize=13, fontweight='bold', color=C_DARK)
ax.set_xlabel('n elementos')
ax.set_ylabel('Tiempo por busqueda (us)')
ax.legend()

# Ejemplo practico: contar categorias
ax = axes[1]
import pandas as pd
categories = np.random.choice(['A', 'B', 'C', 'D', 'E'], size=100000)

# Forma lenta: loop
start = time.perf_counter()
counts_slow = {}
for cat in categories:
    if cat in counts_slow:
        counts_slow[cat] += 1
    else:
        counts_slow[cat] = 1
t_slow = time.perf_counter() - start

# Forma rapida: Counter
from collections import Counter
start = time.perf_counter()
counts_fast = Counter(categories)
t_fast = time.perf_counter() - start

# Forma pandas: value_counts
s = pd.Series(categories)
start = time.perf_counter()
counts_pd = s.value_counts()
t_pd = time.perf_counter() - start

bars = ax.bar(['Loop manual', 'Counter', 'pd.value_counts()'], 
              [t_slow*1000, t_fast*1000, t_pd*1000],
              color=[C_DANGER, C_ORANGE, C_SUCCESS], edgecolor='black')
ax.set_title('Contar categorias (100k elementos)', fontsize=13, fontweight='bold', color=C_DARK)
ax.set_ylabel('Tiempo (ms)')
for bar, t in zip(bars, [t_slow, t_fast, t_pd]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{t*1000:.1f}ms', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Arboles

Los arboles son la base de **Random Forest, XGBoost, Decision Trees, GBDT** — los modelos mas usados en produccion.

### Arbol binario de decision

Cada nodo hace una pregunta: "feature_j > umbral?"
- Si → rama izquierda
- No → rama derecha
- Hoja → prediccion

**Complejidad:**
- Construir el arbol: O(n * m * log n) donde m = features
- Predecir: O(log n) — baja por el arbol, cada nivel descarta la mitad

### Por que Random Forest es rapido para predecir?
- Cada arbol tiene profundidad ~20 → solo 20 comparaciones por prediccion
- 100 arboles × 20 comparaciones = 2,000 operaciones totales
- Para 1 millon de datos, eso es NADA

### Busqueda binaria (misma idea)

Si los datos estan ordenados, puedes buscar en O(log n):
1. Miras el elemento del medio
2. Si es mayor → buscas en la mitad derecha
3. Si es menor → buscas en la mitad izquierda
4. Repites hasta encontrar

Con 1 millon de datos: solo 20 comparaciones (log2(1,000,000) ≈ 20)

In [ ]:
# === Busqueda binaria vs lineal ===
def linear_search(arr, target):
    """O(n): recorre todo"""
    for i, val in enumerate(arr):
        if val == target:
            return i
    return -1

def binary_search(arr, target):
    """O(log n): divide y conquista"""
    left, right = 0, len(arr) - 1
    steps = 0
    while left <= right:
        steps += 1
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid, steps
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1, steps

# Demo
sizes = [100, 1000, 10000, 100000, 1000000]
print(f"{'n datos':>12} {'Lineal (pasos)':>15} {'Binaria (pasos)':>16} {'Speedup':>10}")
print("=" * 58)

for n in sizes:
    arr = np.arange(n)
    target = n - 1  # Peor caso
    
    linear_steps = n  # Peor caso: recorre todo
    _, binary_steps = binary_search(arr, target)
    
    print(f"{n:>12,} {linear_steps:>15,} {binary_steps:>16} {linear_steps/binary_steps:>10,.0f}x")

# Conexion con ML: Decision Tree
print("\nConexion con ML:")
print("Un Decision Tree con max_depth=20 hace como maximo 20 comparaciones.")
print(f"Eso puede separar hasta 2^20 = {2**20:,} grupos distintos.")
print("Por eso max_depth=20 es suficiente para casi cualquier dataset.")

## 5. Sets (Conjuntos)

Basados en hash maps, pero solo almacenan claves (sin valores). Busqueda en O(1).

**Uso en data science:**
- Encontrar valores unicos
- Interseccion de conjuntos (features comunes entre datasets)
- Eliminar duplicados en O(n) en vez de O(n²)

In [ ]:
# === Sets: operaciones de conjuntos ===
# Ejemplo: features de dos datasets
features_train = {'edad', 'peso', 'altura', 'glucosa', 'presion', 'colesterol', 'bmi'}
features_test = {'edad', 'peso', 'altura', 'glucosa', 'presion', 'fumador'}

print("Features train:", features_train)
print("Features test: ", features_test)

print(f"\nComunes (interseccion):     {features_train & features_test}")
print(f"Solo en train (diferencia): {features_train - features_test}")
print(f"Solo en test:               {features_test - features_train}")
print(f"Todas (union):              {features_train | features_test}")

# Velocidad: duplicados
arr = np.random.randint(0, 1000, size=100000)

start = time.perf_counter()
unicos_set = set(arr.tolist())
t_set = time.perf_counter() - start

start = time.perf_counter()
unicos_np = np.unique(arr)
t_np = time.perf_counter() - start

print(f"\nEliminar duplicados (100k elementos):")
print(f"  set():      {t_set*1000:.2f}ms — O(n)")
print(f"  np.unique(): {t_np*1000:.2f}ms — O(n log n) internamente")
print(f"  Unicos encontrados: {len(unicos_set)}")

## 6. Colas y Pilas (Queues y Stacks)

| Estructura | Orden | Uso en ML/DS |
|---|---|---|
| **Stack (LIFO)** | Ultimo en entrar, primero en salir | Backpropagation (recorre capas en reversa), undo/redo |
| **Queue (FIFO)** | Primero en entrar, primero en salir | Pipelines de datos, BFS en grafos |
| **Priority Queue** | Sale el de mayor/menor prioridad | Beam search en NLP, A*, Huffman coding |
| **Deque** | Insercion/eliminacion O(1) en ambos extremos | Sliding windows (como en nuestro proyecto C-MAPSS) |

In [ ]:
# === Deque: ventana deslizante eficiente ===
from collections import deque

# Simular sliding window como en el proyecto de mantenimiento predictivo
sensor_data = np.random.randn(100)  # 100 ciclos de un sensor
window_size = 5

# Forma ineficiente: slice en cada paso
start = time.perf_counter()
means_slow = []
for i in range(window_size, len(sensor_data)):
    window = sensor_data[i-window_size:i]  # Copia O(k) cada vez
    means_slow.append(np.mean(window))
t_slow = time.perf_counter() - start

# Forma eficiente: deque
start = time.perf_counter()
window = deque(maxlen=window_size)
means_fast = []
running_sum = 0
for i, val in enumerate(sensor_data):
    if len(window) == window_size:
        running_sum -= window[0]  # Resta el que sale
    window.append(val)
    running_sum += val             # Suma el que entra
    if len(window) == window_size:
        means_fast.append(running_sum / window_size)
t_fast = time.perf_counter() - start

# Forma pandas (la que realmente usamos)
import pandas as pd
start = time.perf_counter()
means_pd = pd.Series(sensor_data).rolling(window_size).mean().dropna().values
t_pd = time.perf_counter() - start

print("Sliding window (media movil, ventana=5, 100 datos):")
print(f"  Slice cada vez: {t_slow*1e6:.0f}us")
print(f"  Deque O(1):     {t_fast*1e6:.0f}us")
print(f"  pd.rolling():   {t_pd*1e6:.0f}us")
print(f"\nResultados iguales: {np.allclose(means_slow, means_fast)}")
print("\nConexion: pd.rolling() que usamos en el proyecto C-MAPSS")
print("usa internamente una ventana deslizante optimizada.")

## 7. Grafos

Los grafos modelan **relaciones** entre entidades. Nodos + aristas.

**Donde aparecen en ML:**
- **Redes neuronales** son grafos dirigidos aciclicos (DAGs)
- **Graph Neural Networks (GNN)** operan directamente sobre grafos
- **PageRank** de Google: grafo de paginas web
- **Redes sociales:** deteccion de comunidades
- **Knowledge graphs:** relaciones entre conceptos
- **DAGs en MLOps:** pipelines de datos (Airflow, Kubeflow)

### Algoritmos de grafos utiles

| Algoritmo | Que hace | Complejidad | Uso |
|---|---|---|---|
| BFS | Recorre nivel por nivel | O(V+E) | Camino mas corto sin pesos |
| DFS | Recorre en profundidad | O(V+E) | Deteccion de ciclos |
| Dijkstra | Camino mas corto con pesos | O(V log V) | Rutas, redes |
| Topological Sort | Ordena dependencias | O(V+E) | Pipelines, compilacion |